# Model A — Final XGBoost + SMOTE Model
### Individual Contribution: Jianhui

**Objective:** develop, evaluate and register the final Tree-Based Model A using the milestone baseline as the reproducible benchmark.

The milestone baseline is retained unchanged as a benchmark. The final stage then performs a controlled, validation-only candidate comparison, freezes the operating threshold before test evaluation, retrains the selected configuration on the combined development data, and evaluates the untouched test set once.

**Final Model A deliverables**
1. Reproduce the milestone XGBoost + SMOTE baseline.
2. Compare a small, controlled set of candidate configurations using training/validation data only.
3. Select the final feature/configuration and operating threshold without using the held-out test set.
4. Retrain the selected configuration on the combined development data.
5. Evaluate the final model once on the untouched test set.
6. Export final prediction, fairness, feature-importance and deployment artefacts.
7. Log and register the final model in MLflow with a signature and input example.
8. Verify that the registered model reproduces the in-memory model probabilities.

**Leakage controls**
- Stratified train/validation/test split is performed before target-dependent transformations.
- Training target encoding uses out-of-fold values.
- Validation/test target encoding uses a mapping fitted from training data only.
- Final development target encoding is refitted using the combined development data without using the held-out test labels.
- SMOTE is applied only to the training data or combined development data, never to validation/test data.
- The held-out test set is not used for model, feature or threshold selection.

**Relationship to Feature Engineering:** Model A consumes the Feature Engineering hand-off and independently performs the modelling-stage split and target encoding. The milestone baseline is expected to reproduce the FE benchmark. The final model is a separate stage: because it is retrained on the combined development data after validation-only selection, its final test metrics are not expected to equal the milestone baseline metrics.


## 1. Environment and Pipeline Configuration

Package versions are pinned before anything else runs, so this notebook's
MLflow logging behaviour doesn't silently drift if a shared environment's
default package versions change between runs — the exact class of
reproducibility risk that caused Model A's results to diverge from Feature
Engineering's in an earlier iteration of this pipeline.

Configuration follows the same S3-first, local-failover pattern used
throughout this project: every input and output path is declared once, as
a single project root, so this notebook's data source, Feature
Engineering's hand-off location, and Model A's own output location are all
derived consistently rather than hard-coded separately. MLflow is
initialised through the shared `mlflow_utils.initialize_mlflow()` helper,
targeting the same experiment used across EDA, Feature Engineering, and
this notebook, so every stage of the pipeline is queryable from one place.

In [1]:
# ============================================================
# Install required packages
# ============================================================
# Run this cell in a fresh kernel before importing MLflow.
%pip install -q -U \
    "boto3" \
    "botocore" \
    "mlflow==3.15.1" \
    "mlflow-skinny==3.15.1" \
    "mlflow-tracing==3.15.1" \
    "sagemaker-mlflow==0.5.0" \
    "imbalanced-learn" \
    "xgboost" \
    "pyarrow"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# Configuration, S3/local fallback, MLflow
# ============================================================
from pathlib import Path
import os
import json
import warnings
import boto3
from urllib.parse import urlparse

# ============================================================
# Project storage configuration
# Declare ONE S3 project root and ONE local project root.
# Filenames are declared separately and joined only when needed.
# ============================================================

S3_BUCKET_URI = os.getenv(
    "TEAM04_S3_BUCKET_URI",
    "s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/"
).rstrip("/") + "/"

LOCAL_DATA_DIR = Path(
    os.getenv("TEAM04_LOCAL_DATA_DIR", "./data")
)

# ---- Filenames: no filenames are embedded in S3 directory URIs ----
DATA_FILENAME = os.getenv(
    "TEAM04_FRAUD_TRAIN_FILENAME",
    "fraudTrain_sample.csv"
)

EDA_FEATURE_CONFIG_FILENAME = "eda_feature_candidates.json"
FAIRNESS_BASELINE_FILENAME = "fairness_baselines.csv"
FE_OUTPUT_FILENAME = "feature_engineered_model_input.parquet"
FE_CONFIG_FILENAME = "feature_engineering_manifest.json"
MODELA_RESULTS_FILENAME = "model_a_baseline_results.json"
MODELA_FILENAME = "model_a_xgboost_baseline.json"

# ---- S3 directories derived from the single project root ----
S3_RAW_URI = S3_BUCKET_URI + "raw/"
S3_EDA_OUTPUT_URI = S3_BUCKET_URI + "processed/eda/"
S3_FE_OUTPUT_URI = S3_BUCKET_URI + "processed/feature_engineering/"
S3_MODELA_OUTPUT_URI = S3_BUCKET_URI + "processed/modela_baseline/"

# ---- Local directories mirror the S3 structure ----
LOCAL_RAW_DIR = LOCAL_DATA_DIR / "raw"
LOCAL_EDA_OUTPUT_DIR = LOCAL_DATA_DIR / "eda"
LOCAL_FE_OUTPUT_DIR = LOCAL_DATA_DIR / "feature_engineering"
LOCAL_MODELA_OUTPUT_DIR = LOCAL_DATA_DIR / "modela_baseline"

for _p in [
    LOCAL_RAW_DIR,
    LOCAL_EDA_OUTPUT_DIR,
    LOCAL_FE_OUTPUT_DIR,
    LOCAL_MODELA_OUTPUT_DIR,
]:
    _p.mkdir(parents=True, exist_ok=True)


def s3_join(prefix, *parts):
    """Safely join an S3 URI/prefix with one or more path components."""
    return prefix.rstrip("/") + "/" + "/".join(
        str(part).strip("/") for part in parts
    )


def parse_s3_uri(uri):
    p = urlparse(uri)
    if p.scheme != "s3" or not p.netloc:
        raise ValueError(f"Invalid S3 URI: {uri}")
    return p.netloc, p.path.lstrip("/")


def download_s3_to_local(s3_uri, local_path):
    """Try S3 first. If it fails, use an existing local file as failover."""
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    bucket, key = parse_s3_uri(s3_uri)

    try:
        boto3.client("s3").download_file(bucket, key, str(local_path))
        print(f"✓ Loaded from S3: {s3_uri}")
        return local_path
    except Exception as exc:
        if local_path.exists():
            print(f"⚠ S3 unavailable; using local failover: {local_path}")
            print(f"  S3 error: {type(exc).__name__}: {exc}")
            return local_path

        raise RuntimeError(
            f"S3 load failed and local failover does not exist: {local_path}\n"
            f"S3 URI: {s3_uri}\n"
            f"Original error: {type(exc).__name__}: {exc}"
        ) from exc


def upload_local_to_s3(local_path, s3_uri):
    """Best-effort S3 upload. Local output remains the failover copy."""
    local_path = Path(local_path)
    bucket, key = parse_s3_uri(s3_uri)

    try:
        boto3.client("s3").upload_file(str(local_path), bucket, key)
        print(f"✓ Saved to S3: {s3_uri}")
        return True
    except Exception as exc:
        print(f"⚠ S3 upload failed; local output retained: {local_path}")
        print(f"  S3 error: {type(exc).__name__}: {exc}")
        return False


def load_json_s3_or_local(s3_uri, local_path):
    local_path = download_s3_to_local(s3_uri, local_path)
    with open(local_path, "r", encoding="utf-8") as f:
        return json.load(f)

# ---  Environment Setup & MLflow Initialization ---
import pandas as pd
import numpy as np
import warnings

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import mlflow
from mlflow import MlflowClient

from mlflow_utils import initialize_mlflow

warnings.filterwarnings("ignore")

# Run-specific MLflow configuration
STUDENT_ID = "S402"
EXPERIMENT_NAME = "ITI113/team04/ModelA"

# Initialize MLflow and generate a fresh UI link
MLFLOW_APP_ARN = initialize_mlflow(
    student_id=STUDENT_ID,
    experiment_name=EXPERIMENT_NAME
)

print(f"MLflow App ARN: {MLFLOW_APP_ARN}")
print(f"Student ID: {STUDENT_ID}")
print(f"Experiment: {EXPERIMENT_NAME}")

# ---- Derived file paths ----
DATA_S3_URI = s3_join(S3_RAW_URI, DATA_FILENAME)
LOCAL_DATA_PATH = LOCAL_RAW_DIR / DATA_FILENAME

LOCAL_EDA_CONFIG_PATH = LOCAL_EDA_OUTPUT_DIR / EDA_FEATURE_CONFIG_FILENAME
LOCAL_FAIRNESS_PATH = LOCAL_EDA_OUTPUT_DIR / FAIRNESS_BASELINE_FILENAME

LOCAL_FE_OUTPUT_PATH = LOCAL_FE_OUTPUT_DIR / FE_OUTPUT_FILENAME
LOCAL_FE_CONFIG_PATH = LOCAL_FE_OUTPUT_DIR / FE_CONFIG_FILENAME

LOCAL_MODELA_RESULTS_PATH = LOCAL_MODELA_OUTPUT_DIR / MODELA_RESULTS_FILENAME
LOCAL_MODELA_PATH = LOCAL_MODELA_OUTPUT_DIR / MODELA_FILENAME

print("S3 project root:", S3_BUCKET_URI)
print("Local project root:", LOCAL_DATA_DIR)
print("S3 raw directory:", S3_RAW_URI)
print("S3 EDA output directory:", S3_EDA_OUTPUT_URI)
print("S3 Feature Engineering output directory:", S3_FE_OUTPUT_URI)
print("S3 MODELA output directory:", S3_MODELA_OUTPUT_URI)
print("Dataset filename:", DATA_FILENAME)
print("Dataset S3 URI:", DATA_S3_URI)
print("MLflow experiment:", EXPERIMENT_NAME)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    average_precision_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import mlflow
import mlflow.xgboost
from mlflow.models import infer_signature

RANDOM_STATE = 42
TEST_SIZE = 0.20
SMOOTHING = 20
N_SPLITS_TE = 5
FINAL_THRESHOLD = None

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
mlflow.set_experiment(EXPERIMENT_NAME)

print("Feature Engineering input S3 URI:", s3_join(S3_FE_OUTPUT_URI, FE_OUTPUT_FILENAME))
print("Feature Engineering local failover:", LOCAL_FE_OUTPUT_PATH)
print("MODELA output S3 directory:", S3_MODELA_OUTPUT_URI)


Initializing SageMaker MLflow connection for S402...
Target Experiment: ITI113/team04/ModelA
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-ANFQ3RACFV2G
MLflow Tracking URI successfully set.
Fresh MLflow UI URL:
https://app-ANFQ3RACFV2G.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6Ik5HS1VRNiIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNENITVhhSGpOZ0NuTE1nQVV6Sy85bnAvajFJZFMwazZvKzFOL2pJb2FJaU1BWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGdFNucEdlbGhwV25oNWVFNHlPR2xNYlhCV1FuWjVaRk5PZUhOeFZtcDBVRkJHTVVKSVVVRmtWWGxFU25GR1NqbHFiMU14T1hFMmFrMXNPVWszT1ZGd2R6MDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFhN0Q5MTJUOUthdnhKeENLWEFBMUtrQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF3OXN5a

## 2. Load Complete Dataset

The complete Feature Engineering output is used — no row sampling is
applied — so this baseline is evaluated on the same population EDA and
Feature Engineering already validated, rather than a subset that could
behave differently.

In [3]:
# ============================================================
# Load Feature Engineering output — S3 first, local failover
# ============================================================
FE_S3_URI = s3_join(S3_FE_OUTPUT_URI, FE_OUTPUT_FILENAME)
FE_CONFIG_S3_URI = s3_join(S3_FE_OUTPUT_URI, FE_CONFIG_FILENAME)

# Download the EDA-driven feature-engineering output. No CSV filename is embedded in the URI.
FE_PATH = download_s3_to_local(FE_S3_URI, LOCAL_FE_OUTPUT_PATH)
FE_CONFIG = load_json_s3_or_local(FE_CONFIG_S3_URI, LOCAL_FE_CONFIG_PATH)

model_df = pd.read_parquet(FE_PATH)

print(f"Feature-engineered dataset shape: {model_df.shape[0]:,} rows x {model_df.shape[1]} columns")
print("EDA-selected features:", FE_CONFIG["selected_features"])
print("Columns supplied to Model A:", list(model_df.columns))
print(f"Fraud transactions: {model_df['is_fraud'].sum():,}")
print(f"Fraud rate: {model_df['is_fraud'].mean()*100:.4f}%")


✓ Loaded from S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/feature_engineering/feature_engineered_model_input.parquet
✓ Loaded from S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/feature_engineering/feature_engineering_manifest.json
Feature-engineered dataset shape: 1,296,675 rows x 12 columns
EDA-selected features: ['amt', 'amt_log', 'category_te', 'distance_km', 'distance_log', 'city_pop', 'trans_hour', 'day_of_week', 'is_weekend', 'age', 'gender_binary']
Columns supplied to Model A: ['amt', 'amt_log', 'category', 'distance_km', 'distance_log', 'city_pop', 'trans_hour', 'day_of_week', 'is_weekend', 'age', 'gender', 'is_fraud']
Fraud transactions: 7,506
Fraud rate: 0.5789%


**Finding:** the Feature Engineering hand-off loads at 1,296,675 rows,
containing the 11 EDA-selected features plus `is_fraud`, with 7,506 fraud
transactions (0.5789%) — matching EDA and Feature Engineering's own
reported figures exactly. This is the first of several checks in this
notebook confirming Model A is working from the same data Feature
Engineering validated, not a divergent copy.

## 3. Use the EDA → Feature Engineering Hand-off

Model A does not hard-code the EDA feature selection. Feature Engineering
reads `eda_feature_candidates.json` and writes the resulting non-target-
dependent feature-engineered dataset; Model A loads that artifact directly
from S3 with local failover, rather than recomputing any of it.

The target-dependent merchant encoding is intentionally **not**
precomputed in Feature Engineering's output — encoding it before this
notebook's own train/validation/test split would let information leak
across partitions differently than intended here. Model A fits it after
its own split, using out-of-fold training values and a training-only
mapping for validation/test (Section 5).

**Why the hand-off is validated, not just loaded:** the required-column
check below exists specifically to catch a mismatch between what
`FE_CONFIG` declares as selected and what the parquet actually contains —
the two are produced by the same run but read independently here, so a
stale or partially-updated artifact pair would otherwise fail with a
confusing error deep inside feature-matrix construction rather than a
clear message at load time.

In [4]:
# Validate the required hand-off columns before modelling.
required = {"is_fraud"}
if "category_te" in FE_CONFIG["selected_features"]:
    required.add("category")
if "gender_binary" in FE_CONFIG["selected_features"]:
    required.add("gender")
missing = sorted(required - set(model_df.columns))
if missing:
    raise ValueError(f"Feature Engineering output is missing required Model A columns: {missing}")

print("✓ Feature Engineering hand-off validated.")


✓ Feature Engineering hand-off validated.


**Finding:** the hand-off validates successfully — both `category`
(required for `category_te`) and `gender` (required for `gender_binary`)
are present in the loaded parquet, matching what `FE_CONFIG` declares as
selected.

## 4. Stratified Train / Validation / Test Split

A 60/20/20 stratified split is used, with the identical `random_state` and
split ratios Feature Engineering uses on the same source data. This is a
deliberate choice, not a coincidence: it means Model A's split should
reproduce the same row partition FE itself worked with, making later
metric comparisons between the two notebooks meaningful rather than
comparing two different underlying evaluation sets.

In [5]:
X = model_df.drop(columns=["is_fraud"])
y = model_df["is_fraud"].astype(int)

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    stratify=y_dev,
    random_state=RANDOM_STATE
)

print(f"Train: {len(X_train):,} rows | fraud={y_train.mean()*100:.4f}%")
print(f"Valid: {len(X_valid):,} rows | fraud={y_valid.mean()*100:.4f}%")
print(f"Test : {len(X_test):,} rows | fraud={y_test.mean()*100:.4f}%")

Train: 778,005 rows | fraud=0.5789%
Valid: 259,335 rows | fraud=0.5788%
Test : 259,335 rows | fraud=0.5788%


**Finding:** the split produces 778,005 / 259,335 / 259,335 rows for
train/validation/test, with fraud rates of 0.5789%, 0.5788%, and 0.5788% —
identical to Feature Engineering's own split on the same data. This
confirms the two notebooks are working from the same partitioning, not
just similar-looking numbers by chance.

## 5. Leakage-Controlled Target Encoding

`category_te` is target-dependent and is therefore fitted only after the
data split, using the same out-of-fold methodology as Feature Engineering:
training rows are encoded using a mapping fit on the *other* folds of the
training data, so no row's encoded value is ever influenced by its own
label. Validation and test records are encoded using a single mapping
fitted on the complete training partition only.

In [6]:
def fit_te_mapping(categories, target, smoothing=20):
    temp = pd.DataFrame({
        "category": categories,
        "target": target
    })

    global_mean = target.mean()

    stats = (
        temp.groupby("category")["target"]
        .agg(["count", "mean"])
    )

    stats["encoded"] = (
        (stats["count"] * stats["mean"]
         + smoothing * global_mean)
        / (stats["count"] + smoothing)
    )

    return stats["encoded"].to_dict(), global_mean


def apply_te(categories, mapping, global_mean):
    return (
        categories
        .map(mapping)
        .fillna(global_mean)
        .astype(float)
    )


def make_oof_te(
    X_train,
    y_train,
    n_splits=5,
    smoothing=20,
    random_state=42
):
    encoded = pd.Series(
        index=X_train.index,
        dtype=float
    )

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for fit_pos, holdout_pos in skf.split(X_train, y_train):

        fit_idx = X_train.index[fit_pos]
        holdout_idx = X_train.index[holdout_pos]

        mapping, global_mean = fit_te_mapping(
            X_train.loc[fit_idx, "category"],
            y_train.loc[fit_idx],
            smoothing
        )

        encoded.loc[holdout_idx] = apply_te(
            X_train.loc[holdout_idx, "category"],
            mapping,
            global_mean
        )

    return encoded


# Make copies before adding encoded features
X_train = X_train.copy()
X_valid = X_valid.copy()
X_test = X_test.copy()

if "category_te" in FE_CONFIG["selected_features"]:
    # TRAIN: Out-of-Fold Target Encoding
    X_train["category_te"] = make_oof_te(
        X_train, y_train,
        n_splits=N_SPLITS_TE,
        smoothing=SMOOTHING,
        random_state=RANDOM_STATE
    )

    # VALIDATION / TEST: mapping fitted from TRAIN ONLY
    te_mapping, train_global_mean = fit_te_mapping(
        X_train["category"], y_train, smoothing=SMOOTHING
    )
    X_valid["category_te"] = apply_te(X_valid["category"], te_mapping, train_global_mean)
    X_test["category_te"] = apply_te(X_test["category"], te_mapping, train_global_mean)

    print("Target encoding complete.")
    print(f"Training global fraud rate used for smoothing: {train_global_mean:.6f}")
    print(f"Number of learned category mappings: {len(te_mapping)}")
else:
    print("category_te was not selected by EDA; target encoding skipped.")


Target encoding complete.
Training global fraud rate used for smoothing: 0.005789
Number of learned category mappings: 14


**Finding:** the encoder learns mappings for 14 categories against a
training global fraud rate of 0.5789% — matching Feature Engineering's own
encoding step exactly.

## 6. Prepare Model Matrix

Raw categorical values are removed from the XGBoost matrix — `category_te`
replaces raw `category`, and `gender_binary` replaces raw `gender` — since
XGBoost requires numeric input. No standard scaling is applied; tree-based
splits operate on each feature's ordering rather than its scale, so
scaling would add a preprocessing step with no benefit here.

In [7]:
def prepare_matrix(X):
    out = X.copy()

    feature_to_column = {
        "amt": "amt", "amt_log": "amt_log", "category_te": "category_te",
        "distance_km": "distance_km", "distance_log": "distance_log",
        "city_pop": "city_pop", "trans_hour": "trans_hour",
        "day_of_week": "day_of_week", "is_weekend": "is_weekend",
        "age": "age", "gender_binary": "gender_binary"
    }

    if "gender_binary" in FE_CONFIG["selected_features"]:
        out["gender_binary"] = out["gender"].map({"F": 0, "M": 1})

    columns = [feature_to_column[f] for f in FE_CONFIG["selected_features"]]
    return out[columns].apply(pd.to_numeric, errors="coerce")


X_train_model = prepare_matrix(X_train)
X_valid_model = prepare_matrix(X_valid)
X_test_model = prepare_matrix(X_test)

print("Model features:")
display(pd.DataFrame({
    "Feature": X_train_model.columns,
    "Data_Type": X_train_model.dtypes.astype(str)
}))

Model features:


,Feature,Data_Type
amt,amt,float64
amt_log,amt_log,float64
category_te,category_te,float64
distance_km,distance_km,float64
distance_log,distance_log,float64
city_pop,city_pop,int64
trans_hour,trans_hour,int32
day_of_week,day_of_week,int32
is_weekend,is_weekend,int64
age,age,int64


**Finding:** the resulting matrix has exactly 11 columns in the same order
as `FE_CONFIG["selected_features"]`, confirming no feature was dropped,
renamed, or reordered between the hand-off and the model input.

### Pre-Training Feature Power Validation — Gini Impurity

A small CART decision tree is used before the baseline XGBoost model is
trained, to provide an independent Gini-impurity feature-power check using
a different model family than the final classifier. Cross-checking with a
second method matters here specifically: if a single model's importance
score were the only evidence, it would be unclear whether a feature ranking
low reflects the feature itself being weak, or an artifact of that one
model's particular splits. Agreement between two different model types is
stronger evidence than either alone.

This is a diagnostic only; the final baseline model remains XGBoost, whose
own tree-split gain is reported separately below (Section 7).

In [8]:
# Pre-training Gini impurity feature-power validation.
# Original training partition only; SMOTE is not applied here.
from sklearn.tree import DecisionTreeClassifier

gini_probe = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    min_samples_leaf=50,
    random_state=RANDOM_STATE
)

gini_probe.fit(X_train_model, y_train)

gini_importances = pd.Series(
    gini_probe.feature_importances_,
    index=X_train_model.columns,
    name="Gini_Importance"
).sort_values(ascending=False)

print("Top Gini features:")
display(gini_importances.to_frame())

Top Gini features:


,Gini_Importance
amt_log,0.388971
trans_hour,0.287210
category_te,0.215202
age,0.095523
amt,0.013093
distance_log,0.000000
distance_km,0.000000
city_pop,0.000000
day_of_week,0.000000
is_weekend,0.000000


In [9]:
xgb_eval = XGBClassifier(
    n_estimators=50,
    max_depth=5,
    learning_rate=0.05,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
    eval_metric="aucpr"
)

xgb_eval.fit(X_train_model, y_train, verbose=False)

gain_dict = xgb_eval.get_booster().get_score(importance_type="gain")
gain_importances = pd.Series(gain_dict, dtype=float).reindex(X_train_model.columns, fill_value=0.0).sort_values(ascending=False)

display(gain_importances.to_frame("XGBoost_Gain"))


,XGBoost_Gain
amt,545.627869
trans_hour,498.242828
category_te,266.026672
age,229.006607
gender_binary,79.395607
city_pop,32.609688
distance_km,4.088506
day_of_week,2.562734
amt_log,0.000000
distance_log,0.000000


**Finding — the two methods agree on what matters most, with one
informative disagreement.** Both rank `category_te` and `age` among the
top four, and both rank `distance_log` at or near zero — consistent
evidence these are genuinely strong (or genuinely weak) rather than
artifacts of one model's specific splits.

**Where they disagree, and why it's informative rather than
contradictory:** Gini ranks `amt_log` highest (0.389) with `amt` almost
negligible (0.013); Gain ranks `amt` highest (545.6) with `amt_log` at
exactly zero. Since `amt_log` is a deterministic transform of `amt`, the
two features carry the same underlying information — each model simply
picked a different one of the pair to split on first, after which the
other became redundant. This is worth acting on: keeping both `amt` and
`amt_log` in the final feature set doesn't add information, it only
splits a single signal's importance score across two columns and makes
the importance ranking harder to read. Dropping `amt_log` is a reasonable
simplification for the next tuning iteration.

**Also worth noting:** `distance_km`, `city_pop`, `day_of_week`, and
`gender_binary` all show *exactly* zero in the Gini check but small,
non-zero values in the Gain check (e.g. `gender_binary` 79.4). This is
consistent with the Gini probe being a single shallow tree (`max_depth=5`)
with limited capacity to find secondary splits, while the Gain check's
50-tree ensemble has more opportunities to exploit weak, secondary signal.
Neither result is "wrong" — they're measuring feature power under
different amounts of model capacity, which is itself a useful thing to
know when deciding how much complexity the final tuned model needs.

## 7. SMOTE — Training Data Only

SMOTE is applied only to the training partition, using the same
`sampling_strategy=0.20` validated in Feature Engineering — rebalancing
training data to approximately 16.67% fraud while leaving validation and
test data at their natural, untouched distribution. See Feature
Engineering Section 6 for the full rationale on why partial (rather than
full 50/50) rebalancing is used, and why evaluating against the natural
distribution matters for getting a meaningful performance estimate.

In [10]:
print("Before SMOTE:")
display(y_train.value_counts().rename_axis("Class").to_frame("Count"))

smote = SMOTE(
    sampling_strategy=0.20,
    random_state=RANDOM_STATE,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_model,
    y_train
)

print("\nAfter SMOTE:")
display(
    pd.Series(y_train_smote)
    .value_counts()
    .rename_axis("Class")
    .to_frame("Count")
)

print(f"SMOTE training fraud rate: {np.mean(y_train_smote)*100:.2f}%")
print(f"Validation fraud rate: {y_valid.mean()*100:.4f}%")
print(f"Test fraud rate: {y_test.mean()*100:.4f}%")


Before SMOTE:


,Count
Class,
0,773501
1,4504



After SMOTE:


,Count
Class,
0,773501
1,154700


SMOTE training fraud rate: 16.67%
Validation fraud rate: 0.5788%
Test fraud rate: 0.5788%


**Finding:** consistent with the identical split confirmed in Section 4,
the pre-SMOTE training class counts here match Feature Engineering's own
(773,501 legitimate / 4,504 fraud). After resampling, training fraud rate
reaches 16.67%; validation (0.5788%) and test (0.5788%) remain untouched.

## 8. Milestone Baseline — XGBoost + SMOTE

This section deliberately preserves the milestone Model A baseline as the benchmark for the final modelling work. The baseline configuration is not tuned here; it provides a reproducible reference against which the final candidate can be compared.

The baseline uses the agreed XGBoost parameters and SMOTE sampling strategy, selects its operating threshold from validation only, and evaluates the held-out test set once. The baseline artefacts are retained in MLflow so the registry provides a traceable benchmark for the final model.


In [11]:
XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "tree_method": "hist"
}

xgb_model = XGBClassifier(**XGB_PARAMS)

xgb_model.fit(
    X_train_smote,
    y_train_smote,
    eval_set=[(X_valid_model, y_valid)],
    verbose=False
)

print("Baseline Model A training completed.")


Baseline Model A training completed.


**Finding:** training completes successfully on the SMOTE-resampled
training fold, using the identical hyperparameters and training data as
Feature Engineering's own baseline model.

## 9. Validation Evaluation — Baseline Threshold 0.50

PR-AUC is the primary ranking metric because the fraud class is severely
imbalanced (0.58%) — ROC-AUC can look strong even for a weak model under
this much class imbalance, since it's dominated by the large number of
true negatives. Precision, recall and F1 are also reported at the default
0.50 threshold as a reference point, ahead of proper threshold selection
in Section 10.

In [12]:
valid_prob = xgb_model.predict_proba(X_valid_model)[:, 1]
valid_pred_050 = (valid_prob >= 0.50).astype(int)

valid_metrics_050 = {
    "PR-AUC": average_precision_score(y_valid, valid_prob),
    "ROC-AUC": roc_auc_score(y_valid, valid_prob),
    "Precision": precision_score(y_valid, valid_pred_050, zero_division=0),
    "Recall": recall_score(y_valid, valid_pred_050, zero_division=0),
    "F1": f1_score(y_valid, valid_pred_050, zero_division=0),
}

valid_results_050 = pd.DataFrame(
    valid_metrics_050.items(),
    columns=["Metric", "Validation @ 0.50"]
)

display(valid_results_050)

print("Validation confusion matrix @ 0.50:")
print(confusion_matrix(y_valid, valid_pred_050))


,Metric,Validation @ 0.50
0,PR-AUC,0.832986
1,ROC-AUC,0.989561
2,Precision,0.160191
3,Recall,0.917388
4,F1,0.272754


Validation confusion matrix @ 0.50:
[[250615   7219]
 [   124   1377]]


**Finding:** PR-AUC 0.833, Precision 16.0%, Recall 91.7%, F1 0.273 at the
default threshold — **identical to Feature Engineering's own validation
numbers.** The low precision here is not a sign of a weak model (PR-AUC is
strong); it's the expected consequence of training on SMOTE-rebalanced
data (16.67% fraud) and evaluating at a threshold calibrated for that
distribution against validation data's true ~0.58% rate. See Section 10
for the correction.

## 10. Validation Threshold Analysis

The operating threshold is selected using the **validation set only**.
Candidate thresholds are evaluated using precision, recall and F1; the
held-out test set is not used for threshold selection, so that Section
11's evaluation remains a genuine estimate of performance on unseen data.

In [13]:
threshold_rows = []

for threshold in np.arange(0.05, 0.951, 0.05):
    pred = (valid_prob >= threshold).astype(int)
    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(y_valid, pred, zero_division=0),
        "recall": recall_score(y_valid, pred, zero_division=0),
        "f1": f1_score(y_valid, pred, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.sort_values("f1", ascending=False).head(10))

best_row = threshold_df.loc[threshold_df["f1"].idxmax()]
FINAL_THRESHOLD = float(best_row["threshold"])

print(f"Selected validation threshold: {FINAL_THRESHOLD:.2f}")
print(f"Validation Precision: {best_row['precision']:.4f}")
print(f"Validation Recall:    {best_row['recall']:.4f}")
print(f"Validation F1:        {best_row['f1']:.4f}")


,threshold,precision,recall,f1
18,0.95,0.884117,0.726849,0.797806
17,0.90,0.751894,0.793471,0.772123
16,0.85,0.613491,0.830113,0.705549
15,0.80,0.486312,0.852099,0.619221
14,0.75,0.388922,0.870087,0.537559
13,0.70,0.315065,0.884744,0.464661
12,0.65,0.257791,0.892738,0.400060
11,0.60,0.218659,0.904064,0.352147
10,0.55,0.186570,0.910726,0.309696
9,0.50,0.160191,0.917388,0.272754


Selected validation threshold: 0.95
Validation Precision: 0.8841
Validation Recall:    0.7268
Validation F1:        0.7978


**Finding:** F1 peaks at threshold 0.95 (Precision 88.4%, Recall 72.7%,
F1 0.798) — **identical to Feature Engineering's own threshold-selection
result**, both in the selected threshold and the exact metric values. As
noted in Feature Engineering's own analysis: 0.95 is also the top of the
range searched (`0.05`–`0.95`), so this should be treated as a strong
provisional optimum, not a confirmed one — a finer sweep extending past
0.95 would confirm whether performance continues improving beyond it.

## 11. Final Held-Out Test Evaluation

The validation-selected threshold is frozen and applied **once** to the
held-out test set — evaluating it more than once, or adjusting it based on
test performance, would turn the test set into a second validation set and
invalidate it as an estimate of genuinely unseen performance.

In [14]:
TEST_THRESHOLD = FINAL_THRESHOLD

test_prob = xgb_model.predict_proba(X_test_model)[:, 1]
test_pred = (test_prob >= TEST_THRESHOLD).astype(int)

test_metrics = {
    "PR-AUC": average_precision_score(y_test, test_prob),
    "ROC-AUC": roc_auc_score(y_test, test_prob),
    "Precision": precision_score(y_test, test_pred, zero_division=0),
    "Recall": recall_score(y_test, test_pred, zero_division=0),
    "F1": f1_score(y_test, test_pred, zero_division=0),
}

test_results = pd.DataFrame(test_metrics.items(), columns=["Metric", "Test Score"])
display(test_results)

cm = confusion_matrix(y_test, test_pred)
print(f"Test confusion matrix @ threshold {TEST_THRESHOLD:.2f}:")
print(cm)

tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn)
fnr = fn / (fn + tp)
print(f"False Positive Rate: {fpr:.6%}")
print(f"False Negative Rate: {fnr:.6%}")


,Metric,Test Score
0,PR-AUC,0.813346
1,ROC-AUC,0.990449
2,Precision,0.861842
3,Recall,0.698201
4,F1,0.771439


Test confusion matrix @ threshold 0.95:
[[257666    168]
 [   453   1048]]
False Positive Rate: 0.065158%
False Negative Rate: 30.179880%


**Finding:** PR-AUC 0.8133, ROC-AUC 0.9904, Precision 86.18%, Recall
69.82%, F1 0.7714 — **matching Feature Engineering's own final test
metrics exactly**, down to the confusion matrix (TN 257,666 / FP 168 / FN
453 / TP 1,048). This is a meaningful confirmation: it means Model A's
independent re-implementation of the split, encoding, SMOTE, and training
pipeline produces bit-for-bit the same result as Feature Engineering's own
run on the same data, rather than a merely similar one — the two notebooks
are genuinely consistent, not coincidentally close.

**Operational trade-off, unchanged from Feature Engineering's own
finding:** at this threshold, 30.18% of actual fraud is missed (FNR) in
exchange for a very low false-positive rate (0.065%). Whether this is the
right balance is a business decision about the relative cost of missed
fraud versus false alarms, not a purely statistical one.

## 12. MLflow — Log and Register the Milestone Baseline

The milestone baseline is logged and registered separately from the final
Model A candidate. The registry uses a stable model name and a movable
`baseline` alias so reruns create new versions without changing the identity
of the benchmark.

The registered version number is intentionally not hard-coded in the
notebook narrative because it increases whenever the baseline is rerun.


In [15]:
# ============================================================
# Finalise MLflow + export Model A results/model
# ============================================================
from datetime import datetime, timezone

input_example = X_train_model.head(5)
signature = infer_signature(
    X_train_model,
    xgb_model.predict_proba(X_train_model)
)

if mlflow.active_run():
    mlflow.end_run()

RUN_NAME = f"{STUDENT_ID}_MODELA"

with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.set_tags({
        "Course": "ITI113", "Semester": "26S1", "TeamId": "team04",
        "StudentId": STUDENT_ID, "ProjectName": "credit-card-fraud-detection",
        "CreatedByNotebook": "ModelA",
        "PipelineStage": "ModelA_Baseline",
    })
    mlflow.log_params(XGB_PARAMS)
    mlflow.log_param("smote_sampling_strategy", 0.20)
    mlflow.log_param("smote_k_neighbors", 5)
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_param("target_encoding_folds", N_SPLITS_TE)
    mlflow.log_param("target_encoding_smoothing", SMOOTHING)
    mlflow.log_param("operating_threshold", TEST_THRESHOLD)
    mlflow.log_param("feature_engineering_output", FE_S3_URI)

    mlflow.log_metric("validation_pr_auc", valid_metrics_050["PR-AUC"])
    mlflow.log_metric("validation_f1_at_050", valid_metrics_050["F1"])
    mlflow.log_metric("validation_f1_at_selected_threshold", float(best_row["f1"]))
    mlflow.log_metric("test_pr_auc", test_metrics["PR-AUC"])
    mlflow.log_metric("test_f1", test_metrics["F1"])
    mlflow.log_metric("test_precision", test_metrics["Precision"])
    mlflow.log_metric("test_recall", test_metrics["Recall"])
    mlflow.log_metric("test_roc_auc", test_metrics["ROC-AUC"])
    mlflow.log_metric("test_fpr", fpr)
    mlflow.log_metric("test_fnr", fnr)

    # ------------------------------------------------------------
    # Log Model A to MLflow
    # ------------------------------------------------------------
    
    model_info = mlflow.xgboost.log_model(
        xgb_model,
        name="model",
        input_example=input_example,
        signature=signature
    )
    
    print(f"✓ XGBoost model logged to MLflow")
    print(f"Model URI: {model_info.model_uri}")
    
    
    # ------------------------------------------------------------
    # Register Model A in MLflow Model Registry
    # ------------------------------------------------------------
    REGISTERED_MODEL_NAME = "ITI113-team04-ModelA-XGBoost-Baseline"
    
    registered_model = mlflow.register_model(
        model_uri=model_info.model_uri,
        name=REGISTERED_MODEL_NAME
    )
    
    print("✓ Model registered successfully")
    print(f"Registered model: {REGISTERED_MODEL_NAME}")
    print(f"Model version: {registered_model.version}")

    # ------------------------------------------------------------
    # Add description, tags, and alias to the registered model
    # ------------------------------------------------------------
    client = MlflowClient()

    # Registered-model-level description — the overall "model card" summary,
    # shown on the top-level Registered Models page, applies to every version.
    client.update_registered_model(
        name=REGISTERED_MODEL_NAME,
        description=(
            "XGBoost + SMOTE baseline for credit card fraud detection "
            "(Tree-Based Pipeline, Team 04). Trained on 11 leakage-controlled "
            "features from the shared Feature Engineering hand-off, with "
            "out-of-fold category target encoding and training-only SMOTE "
            "resampling. Operating threshold selected from the validation "
            "set only; hyperparameters are un-tuned defaults."
        ),
    )

    # Version-level description — specific to this exact run's results,
    # shown on the Version 1 page.
    client.update_model_version(
        name=REGISTERED_MODEL_NAME,
        version=registered_model.version,
        description=(
            f"Baseline run {RUN_NAME} (run_id={run.info.run_id}). "
            f"Operating threshold {TEST_THRESHOLD}. "
            f"Test PR-AUC {test_metrics['PR-AUC']:.4f}, "
            f"F1 {test_metrics['F1']:.4f}, "
            f"Precision {test_metrics['Precision']:.4f}, "
            f"Recall {test_metrics['Recall']:.4f}."
        ),
    )

    # Version-level tags — structured, filterable metadata (as opposed to
    # the free-text description above).
    client.set_model_version_tag(REGISTERED_MODEL_NAME, registered_model.version, "track", "tree_based_xgboost")
    client.set_model_version_tag(REGISTERED_MODEL_NAME, registered_model.version, "stage", "baseline")
    client.set_model_version_tag(REGISTERED_MODEL_NAME, registered_model.version, "operating_threshold", str(TEST_THRESHOLD))
    client.set_model_version_tag(REGISTERED_MODEL_NAME, registered_model.version, "test_pr_auc", f"{test_metrics['PR-AUC']:.4f}")
    client.set_model_version_tag(REGISTERED_MODEL_NAME, registered_model.version, "test_f1", f"{test_metrics['F1']:.4f}")
    client.set_model_version_tag(REGISTERED_MODEL_NAME, registered_model.version, "hyperparameters_tuned", "false")

    # Alias — a movable pointer to a specific version, so downstream code
    # can reference "the current baseline" without hard-coding a version
    # number. Reassign this alias (rather than adding a new one) when a
    # future retrained version should take its place.
    client.set_registered_model_alias(
        name=REGISTERED_MODEL_NAME,
        alias="baseline",
        version=registered_model.version,
    )

    print("✓ Description, tags, and alias set on registered model")
    print(f"Alias 'baseline' → version {registered_model.version}")
    
    # ------------------------------------------------------------------
    # Result manifest — schema aligned with Model B's export so both
    # tracks can be loaded and compared with the same merge script.
    # ------------------------------------------------------------------

    # Trace dataset provenance back to what Feature Engineering actually
    # used, rather than this notebook's own (unused) DATA_FILENAME default.
    RESOLVED_DATASET_FILENAME = FE_CONFIG.get("dataset_filename", DATA_FILENAME)

    def _rekey_metrics(metrics_dict):
        key_map = {
            "PR-AUC": "pr_auc", "ROC-AUC": "roc_auc",
            "Precision": "precision", "Recall": "recall", "F1": "f1",
        }
        return {key_map.get(k, k): float(v) for k, v in metrics_dict.items()}

    # Full feature importance — all 11 features are recorded even if some are near-zero.
    gini_full = {k: float(v) for k, v in gini_importances.to_dict().items()}
    gain_full = {k: float(v) for k, v in gain_importances.to_dict().items()}

    results_manifest = {
        "schema_version": "1.0",
        "team": "ITI113 Team 04 — Credit Card Fraud Detection",
        "owner": "Jianhui",
        "student_id": STUDENT_ID,
        "track": "Tree-Based Pipeline (Model A)",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "mlflow_experiment": EXPERIMENT_NAME,
        "run_id": run.info.run_id,
        "dataset_filename": RESOLVED_DATASET_FILENAME,
        "feature_engineering_output": FE_S3_URI,
        "feature_count": int(X_train_model.shape[1]),
        "feature_columns": list(X_train_model.columns),
        "dataset": {
            "train_rows": int(len(X_train_model)),
            "valid_rows": int(len(X_valid_model)),
            "test_rows": int(len(X_test_model)),
            "train_fraud_rate": float(y_train.mean()),
            "valid_fraud_rate": float(y_valid.mean()),
            "test_fraud_rate": float(y_test.mean()),
        },
        "models": {
            "xgboost": {
                "mlflow_run_id": run.info.run_id,
                "mlflow_run_name": RUN_NAME,
                "model_registry": {
                    "registered_model_name": REGISTERED_MODEL_NAME,
                    "model_version": str(registered_model.version),
                    "model_uri": model_info.model_uri,
                },
                "hyperparameters": {
                    **XGB_PARAMS,
                    "smote_sampling_strategy": 0.20,
                    "smote_k_neighbors": 5,
                    "target_encoding_folds": N_SPLITS_TE,
                    "target_encoding_smoothing": SMOOTHING,
                },
                "selected_threshold": float(TEST_THRESHOLD),
                "validation_metrics_at_050": _rekey_metrics(valid_metrics_050),
                "validation_metrics_at_selected_threshold": {
                    "precision": float(best_row["precision"]),
                    "recall": float(best_row["recall"]),
                    "f1": float(best_row["f1"]),
                },
                "test_metrics_at_selected_threshold": _rekey_metrics(test_metrics),
                "confusion_matrix": {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)},
                "feature_power_validation": {
                    "gini_top": gini_full,
                    "xgboost_gain_top": gain_full,
                },
            }
        },
    }

    xgb_model.save_model(str(LOCAL_MODELA_PATH))
    upload_local_to_s3(LOCAL_MODELA_PATH, s3_join(S3_MODELA_OUTPUT_URI, MODELA_FILENAME))

    with open(LOCAL_MODELA_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(results_manifest, f, indent=2)

    upload_local_to_s3(LOCAL_MODELA_RESULTS_PATH, s3_join(S3_MODELA_OUTPUT_URI, MODELA_RESULTS_FILENAME))
    mlflow.log_artifact(str(LOCAL_MODELA_RESULTS_PATH), artifact_path="modela_outputs")

    print(f"✓ Model A Baseline MLflow run completed.")
    print(f"Run name: {RUN_NAME}")
    print(f"Run ID: {run.info.run_id}")
    print("Local results:", LOCAL_MODELA_RESULTS_PATH)
    print("S3 results:", s3_join(S3_MODELA_OUTPUT_URI, MODELA_RESULTS_FILENAME))

✓ XGBoost model logged to MLflow
Model URI: models:/m-d2fb2b2a12284d07a07a00966f8901ea


Registered model 'ITI113-team04-ModelA-XGBoost-Baseline' already exists. Creating a new version of this model...
2026/08/18 05:59:07 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ITI113-team04-ModelA-XGBoost-Baseline, version 8
Created version '8' of model 'ITI113-team04-ModelA-XGBoost-Baseline'.


✓ Model registered successfully
Registered model: ITI113-team04-ModelA-XGBoost-Baseline
Model version: 8
✓ Description, tags, and alias set on registered model
Alias 'baseline' → version 8
✓ Saved to S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_xgboost_baseline.json
✓ Saved to S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_baseline_results.json
✓ Model A Baseline MLflow run completed.
Run name: S402_MODELA
Run ID: 9e45051bb2774c3c94ec670670f2f220
Local results: data/modela_baseline/model_a_baseline_results.json
S3 results: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_baseline_results.json
🏃 View run S402_MODELA at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/4/runs/9e45051bb2774c3c94ec670670f2f220
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/4


**Finding:** the milestone baseline is retained in MLflow as a versioned
benchmark under `ITI113-team04-ModelA-XGBoost-Baseline`. The exact version is
determined by the registry at execution time and is tracked by the `baseline`
alias; the notebook does not assume a fixed version number.

The baseline remains separate from the final Model A registration so the
registry records a clear benchmark-to-final progression.


## 12a. Export Predictions for Governance / Fairness Audit

The encoded test set together with the model's actual predictions
(`y_true`, `y_prob`, `y_pred`) is exported separately from the MLflow run,
specifically so the Bias Detection & Fairness Blueprint notebook can
load Model A's real predictions directly — computing Demographic Parity
and Equalized Odds against what the model actually predicted, rather than
needing to reconstruct the split and re-run inference independently.

Before export, the fairness audit dataset is validated to ensure that the required audit columns are present and that the exported ground-truth labels, predicted probabilities, and predictions match the Model A test set.

In [16]:
# ============================================================
# Export Model A predictions for Fairness Audit
# ============================================================

FAIRNESS_INPUT_FILENAME = "model_a_fairness_test_predictions.parquet"

LOCAL_FAIRNESS_OUTPUT_PATH = (
    LOCAL_MODELA_OUTPUT_DIR / FAIRNESS_INPUT_FILENAME
)

fairness_audit_df = X_test.copy()

fairness_audit_df["y_true"] = np.asarray(y_test)
fairness_audit_df["y_prob"] = test_prob
fairness_audit_df["y_pred"] = test_pred

REQUIRED_FAIRNESS_COLUMNS = [
    "age",
    "gender",
    "amt",
    "y_true",
    "y_prob",
    "y_pred",
]

missing_fairness_columns = [
    col
    for col in REQUIRED_FAIRNESS_COLUMNS
    if col not in fairness_audit_df.columns
]

if missing_fairness_columns:
    raise ValueError(
        "Model A fairness export is missing required columns: "
        f"{missing_fairness_columns}"
    )

if len(fairness_audit_df) != len(y_test):
    raise ValueError(
        "Fairness export row count does not match the Model A test set."
    )

if not np.array_equal(
    fairness_audit_df["y_true"].to_numpy(),
    np.asarray(y_test)
):
    raise ValueError(
        "Fairness export y_true does not match Model A y_test."
    )

if not np.allclose(
    fairness_audit_df["y_prob"].to_numpy(),
    test_prob
):
    raise ValueError(
        "Fairness export y_prob does not match Model A test probabilities."
    )

if not np.array_equal(
    fairness_audit_df["y_pred"].to_numpy(),
    test_pred
):
    raise ValueError(
        "Fairness export y_pred does not match Model A test predictions."
    )

print("✓ Fairness export validation passed.")

fairness_audit_df.to_parquet(
    LOCAL_FAIRNESS_OUTPUT_PATH,
    index=False
)

FAIRNESS_S3_URI = s3_join(
    S3_MODELA_OUTPUT_URI,
    FAIRNESS_INPUT_FILENAME
)

upload_local_to_s3(
    LOCAL_FAIRNESS_OUTPUT_PATH,
    FAIRNESS_S3_URI
)

print("✓ Fairness audit input saved locally:")
print(LOCAL_FAIRNESS_OUTPUT_PATH)

print("✓ Fairness audit input saved to S3:")
print(FAIRNESS_S3_URI)

print("\nFairness export columns:")
print(fairness_audit_df.columns.tolist())

✓ Fairness export validation passed.
✓ Saved to S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_fairness_test_predictions.parquet
✓ Fairness audit input saved locally:
data/modela_baseline/model_a_fairness_test_predictions.parquet
✓ Fairness audit input saved to S3:
s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_fairness_test_predictions.parquet

Fairness export columns:
['amt', 'amt_log', 'category', 'distance_km', 'distance_log', 'city_pop', 'trans_hour', 'day_of_week', 'is_weekend', 'age', 'gender', 'category_te', 'y_true', 'y_prob', 'y_pred']


## 13. Milestone Baseline Results Summary

The milestone baseline is retained as the benchmark for the final modelling stage. Its operating threshold was selected from validation only, and its held-out test set was evaluated once.

### Completion checklist
- [x] S3 input with local failover
- [x] Feature Engineering hand-off loaded from S3 with local failover
- [x] Leakage-controlled target encoding
- [x] Gini impurity feature-power validation
- [x] XGBoost tree-split gain validation
- [x] SMOTE applied to training data only
- [x] Validation-only threshold selection
- [x] Held-out test evaluation once
- [x] Baseline MLflow run and model registration
- [x] Baseline test-set prediction export for the milestone fairness audit

### Baseline interpretation

The baseline provides the reproducible reference point for the final Model A work. The final model below is deliberately treated as a separate modelling stage: it may select a different threshold/configuration and is retrained on the combined development data. Therefore, the final test metrics should be compared against the baseline rather than expected to match it exactly.


## 14. Final Model Selection — Validation-Only Tuning

The milestone baseline is retained as the reproducible benchmark. Final Model A
selection is performed using training and validation data only; the held-out
test set is not used to choose features, hyperparameters or the operating
threshold.

A deliberately small candidate set is evaluated to test the main hypothesis
raised by the milestone feature-power analysis: whether retaining both `amt`
and `amt_log` is useful despite their overlapping signal. Each candidate has
its operating threshold selected independently on validation data using the
expanded **0.05–1.00 grid at 0.01 increments**.

Candidates are ranked primarily by validation PR-AUC, with validation F1 used
as the secondary operational criterion. The complete comparison is exported
to CSV and saved to S3 so the selection evidence exists outside the notebook
and can also be logged as an MLflow artefact.


In [17]:
# ============================================================
# Final Model Selection — controlled validation-only candidates
# ============================================================
FINAL_CANDIDATES = [
    {
        "name": "full_11_features_baseline_params",
        "features": list(X_train_model.columns),
        "params": XGB_PARAMS.copy(),
    },
    {
        "name": "drop_amt_log_baseline_params",
        "features": [c for c in X_train_model.columns if c != "amt_log"],
        "params": XGB_PARAMS.copy(),
    },
    {
        "name": "drop_amt_log_deeper",
        "features": [c for c in X_train_model.columns if c != "amt_log"],
        "params": {
            **XGB_PARAMS,
            "max_depth": 5,
            "min_child_weight": 2,
        },
    },
    {
        "name": "drop_amt_log_more_trees",
        "features": [c for c in X_train_model.columns if c != "amt_log"],
        "params": {
            **XGB_PARAMS,
            "n_estimators": 500,
            "max_depth": 5,
        },
    },
]

final_candidate_rows = []
FINAL_THRESHOLD_GRID = np.arange(0.05, 1.001, 0.01)

for candidate in FINAL_CANDIDATES:
    features = candidate["features"]
    params = candidate["params"]

    candidate_model = XGBClassifier(**params)
    candidate_model.fit(
        X_train_smote[features],
        y_train_smote,
        eval_set=[(X_valid_model[features], y_valid)],
        verbose=False,
    )

    candidate_prob = candidate_model.predict_proba(
        X_valid_model[features]
    )[:, 1]

    # Threshold is selected independently for each candidate using validation only.
    candidate_threshold_rows = []
    for threshold in FINAL_THRESHOLD_GRID:
        candidate_pred = (candidate_prob >= threshold).astype(int)
        candidate_threshold_rows.append({
            "threshold": round(float(threshold), 2),
            "precision": precision_score(y_valid, candidate_pred, zero_division=0),
            "recall": recall_score(y_valid, candidate_pred, zero_division=0),
            "f1": f1_score(y_valid, candidate_pred, zero_division=0),
        })

    candidate_threshold_df = pd.DataFrame(candidate_threshold_rows)
    candidate_best = candidate_threshold_df.loc[
        candidate_threshold_df["f1"].idxmax()
    ]

    final_candidate_rows.append({
        "candidate": candidate["name"],
        "feature_count": len(features),
        "features": ", ".join(features),
        "validation_pr_auc": average_precision_score(y_valid, candidate_prob),
        "validation_roc_auc": roc_auc_score(y_valid, candidate_prob),
        "validation_threshold": float(candidate_best["threshold"]),
        "validation_precision": float(candidate_best["precision"]),
        "validation_recall": float(candidate_best["recall"]),
        "validation_f1": float(candidate_best["f1"]),
        "_model": candidate_model,
    })

final_candidate_df = pd.DataFrame(final_candidate_rows)

# Persist the complete candidate comparison outside the notebook.
FINAL_CANDIDATE_COMPARISON_FILENAME = "model_a_final_candidate_comparison.csv"
LOCAL_FINAL_CANDIDATE_COMPARISON_PATH = (
    LOCAL_MODELA_OUTPUT_DIR / FINAL_CANDIDATE_COMPARISON_FILENAME
)
FINAL_CANDIDATE_COMPARISON_S3_URI = s3_join(
    S3_MODELA_OUTPUT_URI,
    FINAL_CANDIDATE_COMPARISON_FILENAME
)

candidate_comparison_export = (
    final_candidate_df
    .drop(columns=["_model"])
    .sort_values(["validation_pr_auc", "validation_f1"], ascending=False)
    .reset_index(drop=True)
)

candidate_comparison_export.to_csv(
    LOCAL_FINAL_CANDIDATE_COMPARISON_PATH,
    index=False
)
upload_local_to_s3(
    LOCAL_FINAL_CANDIDATE_COMPARISON_PATH,
    FINAL_CANDIDATE_COMPARISON_S3_URI
)

print("✓ Final candidate comparison exported:")
print(LOCAL_FINAL_CANDIDATE_COMPARISON_PATH)
print("✓ Final candidate comparison saved to S3:")
print(FINAL_CANDIDATE_COMPARISON_S3_URI)


display(
    final_candidate_df.drop(columns=["_model"])
    .sort_values(["validation_pr_auc", "validation_f1"], ascending=False)
)

best_candidate_row = final_candidate_df.sort_values(
    ["validation_pr_auc", "validation_f1"],
    ascending=False
).iloc[0]

BEST_FINAL_CANDIDATE = best_candidate_row["candidate"]
FINAL_MODEL_FEATURES = [c for c in best_candidate_row["features"].split(", ")]
FINAL_XGB_PARAMS = next(
    c["params"] for c in FINAL_CANDIDATES
    if c["name"] == BEST_FINAL_CANDIDATE
)
FINAL_THRESHOLD = float(best_candidate_row["validation_threshold"])

print(f"Selected final candidate: {BEST_FINAL_CANDIDATE}")
print(f"Selected features ({len(FINAL_MODEL_FEATURES)}): {FINAL_MODEL_FEATURES}")
print(f"Selected validation threshold: {FINAL_THRESHOLD:.2f}")
print(f"Validation PR-AUC: {best_candidate_row['validation_pr_auc']:.4f}")
print(f"Validation F1: {best_candidate_row['validation_f1']:.4f}")


✓ Saved to S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_final_candidate_comparison.csv
✓ Final candidate comparison exported:
data/modela_baseline/model_a_final_candidate_comparison.csv
✓ Final candidate comparison saved to S3:
s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_final_candidate_comparison.csv


,candidate,feature_count,features,validation_pr_auc,validation_roc_auc,validation_threshold,validation_precision,validation_recall,validation_f1
0,full_11_features_baseline_params,11,"amt, amt_log, category_te, distance_km, distan...",0.832986,0.989561,0.94,0.858015,0.748834,0.799715
1,drop_amt_log_baseline_params,10,"amt, category_te, distance_km, distance_log, c...",0.825797,0.985973,0.94,0.870385,0.738175,0.798846
2,drop_amt_log_deeper,10,"amt, category_te, distance_km, distance_log, c...",0.810647,0.987328,0.92,0.849117,0.704863,0.770295
3,drop_amt_log_more_trees,10,"amt, category_te, distance_km, distance_log, c...",0.793253,0.984421,0.96,0.842570,0.698867,0.764020


Selected final candidate: full_11_features_baseline_params
Selected features (11): ['amt', 'amt_log', 'category_te', 'distance_km', 'distance_log', 'city_pop', 'trans_hour', 'day_of_week', 'is_weekend', 'age', 'gender_binary']
Selected validation threshold: 0.94
Validation PR-AUC: 0.8330
Validation F1: 0.7997


## 15. Final Model Retraining on Development Data

After candidate configuration and operating threshold are frozen using
validation results, the selected final XGBoost configuration is retrained on
the combined **train + validation development data**.

Because `category_te` is target-dependent, the development data is re-encoded
using out-of-fold values and a development-only mapping. The held-out test
partition is transformed using that development-only mapping and remains
unseen during model fitting. SMOTE is then applied only to the combined
development matrix.

This retrained model is the model that will be registered and considered for
deployment, not the smaller milestone baseline model.


In [18]:
# ============================================================
# Final retraining on train + validation development data
# ============================================================
X_dev_raw = pd.concat([X_train, X_valid], axis=0)
y_dev_full = pd.concat([y_train, y_valid], axis=0)

X_dev_raw = X_dev_raw.copy()
X_test_final = X_test.copy()

if "category_te" in FE_CONFIG["selected_features"]:
    X_dev_raw["category_te"] = make_oof_te(
        X_dev_raw,
        y_dev_full,
        n_splits=N_SPLITS_TE,
        smoothing=SMOOTHING,
        random_state=RANDOM_STATE,
    )

    dev_te_mapping, dev_global_mean = fit_te_mapping(
        X_dev_raw["category"],
        y_dev_full,
        smoothing=SMOOTHING,
    )

    X_test_final["category_te"] = apply_te(
        X_test_final["category"],
        dev_te_mapping,
        dev_global_mean,
    )

X_dev_model = prepare_matrix(X_dev_raw)
X_test_final_model = prepare_matrix(X_test_final)

# Apply exactly the selected final feature list.
X_dev_final = X_dev_model[FINAL_MODEL_FEATURES].copy()
X_test_final_model = X_test_final_model[FINAL_MODEL_FEATURES].copy()

final_smote = SMOTE(
    sampling_strategy=0.20,
    random_state=RANDOM_STATE,
    k_neighbors=5,
)

X_dev_smote, y_dev_smote = final_smote.fit_resample(
    X_dev_final,
    y_dev_full,
)

print(
    f"Development matrix: {X_dev_final.shape[0]:,} rows x "
    f"{X_dev_final.shape[1]} features"
)
print(
    f"After SMOTE: {X_dev_smote.shape[0]:,} rows | "
    f"fraud={y_dev_smote.mean()*100:.2f}%"
)

final_xgb_model = XGBClassifier(**FINAL_XGB_PARAMS)
final_xgb_model.fit(
    X_dev_smote,
    y_dev_smote,
    verbose=False,
)

print("✓ Final Model A retraining completed.")


Development matrix: 1,037,340 rows x 11 features
After SMOTE: 1,237,602 rows | fraud=16.67%
✓ Final Model A retraining completed.


## 15a. Final Model Feature-Power Cross-Check — Gini

The final model retains the independent Gini-impurity diagnostic used in the
milestone baseline. The probe is fitted on the combined development data
(train + validation) before SMOTE, using the **same final feature set** selected
by validation-only model selection.

This is a diagnostic cross-check, not the deployed model. Comparing final
CART Gini importance with the final XGBoost Gain importance provides evidence
that the final feature ranking is not being interpreted from a single model
family alone. The held-out test set is not used for this diagnostic.


In [19]:

# ============================================================
# Final-model Gini cross-check on combined development data
# ============================================================
from sklearn.tree import DecisionTreeClassifier

final_gini_probe = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    min_samples_leaf=50,
    random_state=RANDOM_STATE,
)

final_gini_probe.fit(X_dev_final, y_dev_full)

final_gini_importances = (
    pd.Series(
        final_gini_probe.feature_importances_,
        index=FINAL_MODEL_FEATURES,
        name="Final_Gini_Importance",
    )
    .sort_values(ascending=False)
)

print("Final Model A Gini cross-check:")
display(final_gini_importances.to_frame())

print(
    "✓ Final Gini probe fitted on combined train+validation development data only."
)


Final Model A Gini cross-check:


,Final_Gini_Importance
amt_log,0.386300
trans_hour,0.290990
category_te,0.216906
age,0.092779
amt,0.013024
distance_log,0.000000
distance_km,0.000000
city_pop,0.000000
day_of_week,0.000000
is_weekend,0.000000


✓ Final Gini probe fitted on combined train+validation development data only.


## 16. Final Held-Out Test Evaluation

The final model is evaluated on the untouched held-out test partition only
after feature selection, candidate selection and threshold selection have been
frozen.

The test threshold is exactly the validation-selected `FINAL_THRESHOLD`.
No test-driven threshold adjustment is performed. These metrics therefore
represent the final Model A estimate on data that was not used for any
modelling decision.


In [20]:
# ============================================================
# Final held-out test evaluation — one final evaluation
# ============================================================
final_test_prob = final_xgb_model.predict_proba(
    X_test_final_model
)[:, 1]

final_test_pred = (
    final_test_prob >= FINAL_THRESHOLD
).astype(int)

final_test_metrics = {
    "PR-AUC": average_precision_score(y_test, final_test_prob),
    "ROC-AUC": roc_auc_score(y_test, final_test_prob),
    "Precision": precision_score(y_test, final_test_pred, zero_division=0),
    "Recall": recall_score(y_test, final_test_pred, zero_division=0),
    "F1": f1_score(y_test, final_test_pred, zero_division=0),
}

final_cm = confusion_matrix(y_test, final_test_pred)
final_tn, final_fp, final_fn, final_tp = final_cm.ravel()
final_fpr = final_fp / (final_fp + final_tn)
final_fnr = final_fn / (final_fn + final_tp)

display(
    pd.DataFrame(
        final_test_metrics.items(),
        columns=["Metric", "Final Test Score"]
    )
)

print(f"Final threshold: {FINAL_THRESHOLD:.2f}")
print("Final confusion matrix:")
print(final_cm)
print(f"Final FPR: {final_fpr:.6%}")
print(f"Final FNR: {final_fnr:.6%}")


,Metric,Final Test Score
0,PR-AUC,0.834790
1,ROC-AUC,0.991000
2,Precision,0.887910
3,Recall,0.738841
4,F1,0.806545


Final threshold: 0.94
Final confusion matrix:
[[257694    140]
 [   392   1109]]
Final FPR: 0.054299%
Final FNR: 26.115923%


## 17. Final Model Validation and Delivery Artefacts

This section creates the persistent artefacts required by the downstream
Fairness, MLOps and group model-comparison work packages.

The final outputs include:

- final held-out test predictions for the fairness audit;
- final XGBoost Gain importance;
- final CART Gini importance using the same final feature set;
- the complete candidate comparison table;
- a baseline-versus-final comparison table;
- a machine-readable final results manifest.

The candidate comparison is exported independently of the notebook and is
also logged to the final MLflow run so the model-selection evidence remains
available if the notebook is not executed.


In [21]:
# ============================================================
# Final Model A artefacts: fairness export, feature importance,
# baseline-vs-final comparison and machine-readable results manifest
# ============================================================

# 1. Final fairness/audit prediction export
FINAL_FAIRNESS_INPUT_FILENAME = "model_a_final_fairness_test_predictions.parquet"
LOCAL_FINAL_FAIRNESS_OUTPUT_PATH = LOCAL_MODELA_OUTPUT_DIR / FINAL_FAIRNESS_INPUT_FILENAME

final_fairness_df = X_test.copy()
final_fairness_df["y_true"] = np.asarray(y_test)
final_fairness_df["y_prob"] = final_test_prob
final_fairness_df["y_pred"] = final_test_pred

REQUIRED_FINAL_FAIRNESS_COLUMNS = [
    "age", "gender", "amt", "y_true", "y_prob", "y_pred"
]
missing_final_fairness_columns = [
    c for c in REQUIRED_FINAL_FAIRNESS_COLUMNS if c not in final_fairness_df.columns
]
if missing_final_fairness_columns:
    raise ValueError(
        "Final Model A fairness export is missing required columns: "
        f"{missing_final_fairness_columns}"
    )

if len(final_fairness_df) != len(y_test):
    raise ValueError("Final fairness export row count does not match y_test.")
if not np.array_equal(final_fairness_df["y_true"].to_numpy(), np.asarray(y_test)):
    raise ValueError("Final fairness export y_true does not match y_test.")
if not np.allclose(final_fairness_df["y_prob"].to_numpy(), final_test_prob):
    raise ValueError("Final fairness export y_prob does not match final test probabilities.")
if not np.array_equal(final_fairness_df["y_pred"].to_numpy(), final_test_pred):
    raise ValueError("Final fairness export y_pred does not match final test predictions.")

final_fairness_df.to_parquet(LOCAL_FINAL_FAIRNESS_OUTPUT_PATH, index=False)
FINAL_FAIRNESS_S3_URI = s3_join(S3_MODELA_OUTPUT_URI, FINAL_FAIRNESS_INPUT_FILENAME)
upload_local_to_s3(LOCAL_FINAL_FAIRNESS_OUTPUT_PATH, FINAL_FAIRNESS_S3_URI)

# 2. Final XGBoost feature importance using native Gain
final_gain_dict = final_xgb_model.get_booster().get_score(importance_type="gain")
final_gain_importances = (
    pd.Series(final_gain_dict, dtype=float)
    .reindex(FINAL_MODEL_FEATURES, fill_value=0.0)
    .sort_values(ascending=False)
)

print("Final Model A feature importance (XGBoost Gain):")
display(final_gain_importances.to_frame("Gain_Importance"))

# 3. Persist final Gini + XGBoost Gain feature importance.
# The Gini probe was fitted earlier on the combined development data
# and provides the independent cross-check required for the final model.
FINAL_FEATURE_IMPORTANCE_FILENAME = "model_a_final_feature_importance.csv"
LOCAL_FINAL_FEATURE_IMPORTANCE_PATH = (
    LOCAL_MODELA_OUTPUT_DIR / FINAL_FEATURE_IMPORTANCE_FILENAME
)
FINAL_FEATURE_IMPORTANCE_S3_URI = s3_join(
    S3_MODELA_OUTPUT_URI,
    FINAL_FEATURE_IMPORTANCE_FILENAME,
)

final_feature_importance_export = pd.DataFrame({
    "feature": FINAL_MODEL_FEATURES,
    "gini_importance": [
        float(final_gini_importances.get(feature, 0.0))
        for feature in FINAL_MODEL_FEATURES
    ],
    "xgboost_gain": [
        float(final_gain_importances.get(feature, 0.0))
        for feature in FINAL_MODEL_FEATURES
    ],
})

final_feature_importance_export = (
    final_feature_importance_export
    .sort_values(
        ["xgboost_gain", "gini_importance"],
        ascending=False
    )
    .reset_index(drop=True)
)

final_feature_importance_export.to_csv(
    LOCAL_FINAL_FEATURE_IMPORTANCE_PATH,
    index=False
)

upload_local_to_s3(
    LOCAL_FINAL_FEATURE_IMPORTANCE_PATH,
    FINAL_FEATURE_IMPORTANCE_S3_URI
)

print("✓ Final feature-importance cross-check exported:")
print(LOCAL_FINAL_FEATURE_IMPORTANCE_PATH)
print("✓ Final feature-importance cross-check saved to S3:")
print(FINAL_FEATURE_IMPORTANCE_S3_URI)

# 4. Baseline vs final comparison
baseline_comparison = pd.DataFrame([
    {
        "Model": "Milestone Baseline",
        "PR-AUC": float(test_metrics["PR-AUC"]),
        "ROC-AUC": float(test_metrics["ROC-AUC"]),
        "Precision": float(test_metrics["Precision"]),
        "Recall": float(test_metrics["Recall"]),
        "F1": float(test_metrics["F1"]),
    },
    {
        "Model": "Final Model A",
        "PR-AUC": float(final_test_metrics["PR-AUC"]),
        "ROC-AUC": float(final_test_metrics["ROC-AUC"]),
        "Precision": float(final_test_metrics["Precision"]),
        "Recall": float(final_test_metrics["Recall"]),
        "F1": float(final_test_metrics["F1"]),
    },
])

print("Baseline vs Final Model A:")
display(baseline_comparison)

comparison_delta = {
    metric: float(baseline_comparison.loc[1, metric] - baseline_comparison.loc[0, metric])
    for metric in ["PR-AUC", "ROC-AUC", "Precision", "Recall", "F1"]
}
print("Final minus baseline metric change:")
print(comparison_delta)

# 5. Final machine-readable results manifest
FINAL_RESULTS_FILENAME = "model_a_final_results.json"
LOCAL_FINAL_RESULTS_PATH = LOCAL_MODELA_OUTPUT_DIR / FINAL_RESULTS_FILENAME

final_results_manifest = {
    "schema_version": "2.0",
    "model_stage": "final",
    "student_id": STUDENT_ID,
    "track": "Tree-Based Pipeline (Model A)",
    "candidate": BEST_FINAL_CANDIDATE,
    "feature_count": int(len(FINAL_MODEL_FEATURES)),
    "feature_columns": list(FINAL_MODEL_FEATURES),
    "selected_threshold": float(FINAL_THRESHOLD),
    "selection_basis": "validation_only",
    "threshold_grid": "0.05_to_1.00_step_0.01",
    "dataset": {
        "train_rows": int(len(X_train)),
        "validation_rows": int(len(X_valid)),
        "test_rows": int(len(X_test)),
        "development_rows": int(len(X_dev_final)),
    },
    "baseline_test_metrics": {k: float(v) for k, v in test_metrics.items()},
    "final_test_metrics": {k: float(v) for k, v in final_test_metrics.items()},
    "final_confusion_matrix": {
        "TN": int(final_tn), "FP": int(final_fp),
        "FN": int(final_fn), "TP": int(final_tp)
    },
    "final_fpr": float(final_fpr),
    "final_fnr": float(final_fnr),
    "validation_selected_metrics": {
        "pr_auc": float(best_candidate_row["validation_pr_auc"]),
        "roc_auc": float(best_candidate_row["validation_roc_auc"]),
        "precision": float(best_candidate_row["validation_precision"]),
        "recall": float(best_candidate_row["validation_recall"]),
        "f1": float(best_candidate_row["validation_f1"]),
    },
    "mlflow": {
        "experiment": EXPERIMENT_NAME,
        "run_name": f"{STUDENT_ID}_MODELA_FINAL",
        "registered_model_name": "ITI113-team04-ModelA-XGBoost-Final",
    },
    "deployment": {
        "input_type": "model_ready_numeric_features",
        "state_available": False,
        "fairness_export": FINAL_FAIRNESS_INPUT_FILENAME,
    },
    "candidate_comparison": {
        "filename": FINAL_CANDIDATE_COMPARISON_FILENAME,
        "s3_uri": FINAL_CANDIDATE_COMPARISON_S3_URI,
        "selection_basis": "validation_pr_auc_then_validation_f1",
        "rows": candidate_comparison_export.to_dict(orient="records"),
    },
    "feature_importance_gini": {
        k: float(v) for k, v in final_gini_importances.to_dict().items()
    },
    "feature_importance_gain": {
        k: float(v) for k, v in final_gain_importances.to_dict().items()
    },
    "feature_importance_artifact": {
        "filename": FINAL_FEATURE_IMPORTANCE_FILENAME,
        "s3_uri": FINAL_FEATURE_IMPORTANCE_S3_URI,
    },
}

with open(LOCAL_FINAL_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(final_results_manifest, f, indent=2)

FINAL_RESULTS_S3_URI = s3_join(S3_MODELA_OUTPUT_URI, FINAL_RESULTS_FILENAME)
upload_local_to_s3(LOCAL_FINAL_RESULTS_PATH, FINAL_RESULTS_S3_URI)

print("✓ Final fairness export:", LOCAL_FINAL_FAIRNESS_OUTPUT_PATH)
print("✓ Final results manifest:", LOCAL_FINAL_RESULTS_PATH)
print("✓ Final Model A artefact validation completed.")


✓ Saved to S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_final_fairness_test_predictions.parquet
Final Model A feature importance (XGBoost Gain):


,Gain_Importance
amt,1647.699219
amt_log,967.364624
category_te,480.393707
is_weekend,245.526230
trans_hour,217.978088
gender_binary,217.643478
day_of_week,160.319031
age,83.569466
city_pop,42.747082
distance_log,41.730881


✓ Saved to S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_final_feature_importance.csv
✓ Final feature-importance cross-check exported:
data/modela_baseline/model_a_final_feature_importance.csv
✓ Final feature-importance cross-check saved to S3:
s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_final_feature_importance.csv
Baseline vs Final Model A:


,Model,PR-AUC,ROC-AUC,Precision,Recall,F1
0,Milestone Baseline,0.813346,0.990449,0.861842,0.698201,0.771439
1,Final Model A,0.834790,0.991000,0.887910,0.738841,0.806545


Final minus baseline metric change:
{'PR-AUC': 0.021444333940039972, 'ROC-AUC': 0.000550915072910052, 'Precision': 0.026068222999452217, 'Recall': 0.04063957361758819, 'F1': 0.03510636731689365}
✓ Saved to S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_final_results.json
✓ Final fairness export: data/modela_baseline/model_a_final_fairness_test_predictions.parquet
✓ Final results manifest: data/modela_baseline/model_a_final_results.json
✓ Final Model A artefact validation completed.


## 18. Deployment Contract and Preprocessing Artefact

The registered XGBoost model accepts a **model-ready numeric feature vector**,
not a raw transaction record. The deployment contract is therefore designed
to be self-contained enough for the downstream SageMaker inference workflow
to reconstruct the exact model input contract.

The contract records:

- registered model name and registry URI/version when available;
- exact feature order and feature count;
- raw fields required before preprocessing;
- deterministic transformations required for model inputs;
- gender mapping;
- development-only target-encoding mapping provenance and smoothing/fold settings;
- operating threshold and decision rule;
- target name;
- dataset / Feature Engineering hand-off provenance;
- explicit statement that `state` is unavailable because it was removed during EDA.

The final feature-importance cross-check is also persisted as a CSV artefact containing both Gini and XGBoost Gain scores.

The companion preprocessing bundle contains the learned target-encoding mapping
and categorical mapping. If the endpoint accepts raw transaction fields, this
preprocessing bundle must execute before the registered XGBoost model.


In [22]:
# ============================================================
# Export deployment contract + preprocessing bundle
# ============================================================

# ------------------------------------------------------------
# Development-only target-encoding provenance sanity check
# ------------------------------------------------------------
# These are the mappings used by the final deployment bundle.
# They must come from the combined train + validation development data,
# not from the earlier milestone training-only mapping.
if "category_te" in FE_CONFIG["selected_features"]:
    expected_dev_global_mean = float(y_dev_full.mean())

    if not np.isclose(
        float(dev_global_mean),
        expected_dev_global_mean,
        rtol=0.0,
        atol=1e-12,
    ):
        raise ValueError(
            "Final deployment target-encoding global mean is not sourced "
            "from the combined train+validation development target."
        )

    if set(dev_te_mapping.keys()) != set(X_dev_raw["category"].unique()):
        raise ValueError(
            "Final deployment target-encoding mapping does not cover the "
            "combined development category values."
        )

    print("✓ Final target-encoding mapping provenance verified.")
    print(
        f"  Development rows used for mapping: {len(X_dev_raw):,}"
    )
    print(
        f"  Development fraud rate / global mean: {dev_global_mean:.8f}"
    )

import joblib

DEPLOYMENT_CONTRACT_FILENAME = "model_a_deployment_contract.json"
PREPROCESSING_BUNDLE_FILENAME = "model_a_preprocessing_bundle.joblib"

deployment_contract = {
    "schema_version": "1.0",
    "model_name": "ModelA_XGBoost_Final",
    "model_type": "xgboost",
    "student_id": STUDENT_ID,
    "experiment": EXPERIMENT_NAME,
    "input_type": "model_ready_numeric_features",
    "raw_input_fields_required": [
        "amt",
        "category",
        "distance_km",
        "distance_log",
        "city_pop",
        "trans_hour",
        "day_of_week",
        "is_weekend",
        "age",
        "gender",
    ],
    "feature_columns": list(FINAL_MODEL_FEATURES),
    "feature_count": len(FINAL_MODEL_FEATURES),
    "feature_engineering_output": FE_S3_URI,
    "feature_engineering_manifest": FE_CONFIG_S3_URI,
    "transformations": {
        "amt": "numeric_passthrough",
        "amt_log": "already_present_in_feature_engineering_hand_off",
        "category_te": "learned_target_encoding_from_combined_train_plus_validation_development_data",
        "distance_km": "numeric_passthrough",
        "distance_log": "already_present_in_feature_engineering_hand_off",
        "city_pop": "numeric_passthrough",
        "trans_hour": "numeric_passthrough",
        "day_of_week": "numeric_passthrough",
        "is_weekend": "numeric_passthrough",
        "age": "numeric_passthrough",
        "gender_binary": "gender_mapping_F_0_M_1",
    },
    "gender_binary_mapping": {"F": 0, "M": 1},
    "target_encoding": {
        "enabled": "category_te" in FE_CONFIG["selected_features"],
        "source": "combined_train_plus_validation_development_data",
        "smoothing": SMOOTHING,
        "folds": N_SPLITS_TE,
        "fallback_global_mean": float(
            dev_global_mean if "category_te" in FE_CONFIG["selected_features"]
            else y_dev_full.mean()
        ),
        "mapping_artifact": PREPROCESSING_BUNDLE_FILENAME,
    },
    "operating_threshold": float(FINAL_THRESHOLD),
    "threshold_selection_basis": "validation_only",
    "decision_rule": "fraud if predicted_probability >= operating_threshold",
    "target_column": "is_fraud",
    "output": {
        "probability": "predicted fraud probability",
        "prediction": "binary fraud decision using operating_threshold",
    },
    "state_available": False,
    "state_note": "state was removed during EDA and is not part of the final model feature set",
    "artifacts": {
        "preprocessing_bundle": PREPROCESSING_BUNDLE_FILENAME,
        "fairness_export": FINAL_FAIRNESS_INPUT_FILENAME,
        "results_manifest": FINAL_RESULTS_FILENAME,
        "candidate_comparison": FINAL_CANDIDATE_COMPARISON_FILENAME,
        "feature_importance": FINAL_FEATURE_IMPORTANCE_FILENAME,
    },
}

deployment_bundle = {
    "feature_columns": FINAL_MODEL_FEATURES,
    "gender_mapping": {"F": 0, "M": 1},
    "category_te_mapping": (
        dev_te_mapping if "category_te" in FE_CONFIG["selected_features"] else {}
    ),
    "category_te_global_mean": (
        float(dev_global_mean)
        if "category_te" in FE_CONFIG["selected_features"]
        else float(y_dev_full.mean())
    ),
    "category_te_smoothing": SMOOTHING,
    "operating_threshold": FINAL_THRESHOLD,
}

LOCAL_DEPLOYMENT_CONTRACT_PATH = (
    LOCAL_MODELA_OUTPUT_DIR / DEPLOYMENT_CONTRACT_FILENAME
)
LOCAL_PREPROCESSING_BUNDLE_PATH = (
    LOCAL_MODELA_OUTPUT_DIR / PREPROCESSING_BUNDLE_FILENAME
)

with open(LOCAL_DEPLOYMENT_CONTRACT_PATH, "w", encoding="utf-8") as f:
    json.dump(deployment_contract, f, indent=2)

joblib.dump(
    deployment_bundle,
    LOCAL_PREPROCESSING_BUNDLE_PATH
)

upload_local_to_s3(
    LOCAL_DEPLOYMENT_CONTRACT_PATH,
    s3_join(S3_MODELA_OUTPUT_URI, DEPLOYMENT_CONTRACT_FILENAME)
)
upload_local_to_s3(
    LOCAL_PREPROCESSING_BUNDLE_PATH,
    s3_join(S3_MODELA_OUTPUT_URI, PREPROCESSING_BUNDLE_FILENAME)
)

print("✓ Deployment contract exported:", LOCAL_DEPLOYMENT_CONTRACT_PATH)
print("✓ Preprocessing bundle exported:", LOCAL_PREPROCESSING_BUNDLE_PATH)


✓ Final target-encoding mapping provenance verified.
  Development rows used for mapping: 1,037,340
  Development fraud rate / global mean: 0.00578884
✓ Saved to S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_deployment_contract.json
✓ Saved to S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_preprocessing_bundle.joblib
✓ Deployment contract exported: data/modela_baseline/model_a_deployment_contract.json
✓ Preprocessing bundle exported: data/modela_baseline/model_a_preprocessing_bundle.joblib


## 19. Final MLflow Registration

The final model is logged separately from the milestone baseline so MLflow
provides a clear progression from benchmark to final Model A candidate.

The final run records the selected candidate, final threshold, validation
selection metrics, held-out test metrics and deployment artefacts. The complete
candidate comparison table, final results manifest, deployment contract and
preprocessing bundle are retained as MLflow artefacts.

The registered model receives the `champion-candidate` alias. This does **not**
mean it is the project champion: Model B must still be evaluated on the same
common basis before the group makes the final model-selection decision.

The MLflow model signature describes the model-ready numeric input contract.


In [26]:
# ============================================================
# Final MLflow run + model registration
# ============================================================
FINAL_RUN_NAME = f"{STUDENT_ID}_MODELA_FINAL"
FINAL_REGISTERED_MODEL_NAME = "ITI113-team04-ModelA-XGBoost-Final"

FINAL_MODEL_FILENAME = "model_a_xgboost_final.json"
LOCAL_FINAL_MODEL_PATH = LOCAL_MODELA_OUTPUT_DIR / FINAL_MODEL_FILENAME

final_input_example = X_dev_final.head(5)
final_signature = infer_signature(
    X_dev_final,
    final_xgb_model.predict_proba(X_dev_final)
)

final_xgb_model.save_model(str(LOCAL_FINAL_MODEL_PATH))
upload_local_to_s3(LOCAL_FINAL_MODEL_PATH, s3_join(S3_MODELA_OUTPUT_URI, FINAL_MODEL_FILENAME))

print(f"✓ Final model saved locally and to S3: {FINAL_MODEL_FILENAME}")

if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name=FINAL_RUN_NAME) as final_run:
    mlflow.set_tags({
        "Course": "ITI113",
        "Semester": "26S1",
        "TeamId": "team04",
        "StudentId": STUDENT_ID,
        "ProjectName": "credit-card-fraud-detection",
        "CreatedByNotebook": "ModelA_Final",
        "PipelineStage": "ModelA_Final",
        "BaselineRunRetained": "true",
        "DeploymentReadyContract": "true",
    })

    for key, value in FINAL_XGB_PARAMS.items():
        mlflow.log_param(key, value)

    mlflow.log_param("smote_sampling_strategy", 0.20)
    mlflow.log_param("smote_k_neighbors", 5)
    mlflow.log_param("target_encoding_folds", N_SPLITS_TE)
    mlflow.log_param("target_encoding_smoothing", SMOOTHING)
    mlflow.log_param("operating_threshold", FINAL_THRESHOLD)
    mlflow.log_param("selected_candidate", BEST_FINAL_CANDIDATE)
    mlflow.log_param("feature_count", len(FINAL_MODEL_FEATURES))

    mlflow.log_metric("validation_pr_auc_selected", float(best_candidate_row["validation_pr_auc"]))
    mlflow.log_metric("validation_f1_selected", float(best_candidate_row["validation_f1"]))
    mlflow.log_metric("test_pr_auc", final_test_metrics["PR-AUC"])
    mlflow.log_metric("test_roc_auc", final_test_metrics["ROC-AUC"])
    mlflow.log_metric("test_precision", final_test_metrics["Precision"])
    mlflow.log_metric("test_recall", final_test_metrics["Recall"])
    mlflow.log_metric("test_f1", final_test_metrics["F1"])
    mlflow.log_metric("test_fpr", final_fpr)
    mlflow.log_metric("test_fnr", final_fnr)

    final_model_info = mlflow.xgboost.log_model(
        final_xgb_model,
        name="model",
        input_example=final_input_example,
        signature=final_signature,
    )

    # MLflow Registry names must match the registry naming rules.
    final_registered_model = mlflow.register_model(
        model_uri=final_model_info.model_uri,
        name=FINAL_REGISTERED_MODEL_NAME,
    )

    client = MlflowClient()

    client.update_registered_model(
        name=FINAL_REGISTERED_MODEL_NAME,
        description=(
            "Final Model A XGBoost + SMOTE model for credit card fraud detection. "
            "Selected using validation-only model and threshold comparison, "
            "then retrained on the combined development data. The held-out test "
            "set was reserved for final evaluation."
        ),
    )

    client.update_model_version(
        name=FINAL_REGISTERED_MODEL_NAME,
        version=final_registered_model.version,
        description=(
            f"Final Model A run {FINAL_RUN_NAME}; threshold={FINAL_THRESHOLD:.2f}; "
            f"test PR-AUC={final_test_metrics['PR-AUC']:.4f}; "
            f"test F1={final_test_metrics['F1']:.4f}."
        ),
    )

    client.set_model_version_tag(
        FINAL_REGISTERED_MODEL_NAME,
        final_registered_model.version,
        "stage",
        "final",
    )
    client.set_model_version_tag(
        FINAL_REGISTERED_MODEL_NAME,
        final_registered_model.version,
        "operating_threshold",
        str(FINAL_THRESHOLD),
    )
    client.set_model_version_tag(
        FINAL_REGISTERED_MODEL_NAME,
        final_registered_model.version,
        "deployment_contract",
        DEPLOYMENT_CONTRACT_FILENAME,
    )

    client.set_registered_model_alias(
        name=FINAL_REGISTERED_MODEL_NAME,
        alias="champion-candidate",
        version=final_registered_model.version,
    )

    mlflow.log_artifact(
        str(LOCAL_DEPLOYMENT_CONTRACT_PATH),
        artifact_path="deployment",
    )
    mlflow.log_artifact(
        str(LOCAL_FINAL_CANDIDATE_COMPARISON_PATH),
        artifact_path="model_selection",
    )
    mlflow.log_artifact(
        str(LOCAL_FINAL_FEATURE_IMPORTANCE_PATH),
        artifact_path="model_selection",
    )
    mlflow.log_artifact(
        str(LOCAL_PREPROCESSING_BUNDLE_PATH),
        artifact_path="deployment",
    )
    mlflow.log_artifact(
        str(LOCAL_FINAL_FAIRNESS_OUTPUT_PATH),
        artifact_path="governance",
    )

    print("✓ Final model logged and registered.")
    print(f"Run ID: {final_run.info.run_id}")
    print(f"Registered model: {FINAL_REGISTERED_MODEL_NAME}")
    print(f"Version: {final_registered_model.version}")
    print(f"Alias: champion-candidate → {final_registered_model.version}")


    # Update the in-memory final results manifest with the exact registered
    # model version/URI now that the registry operation has completed.
    final_results_manifest["mlflow"]["registered_model_version"] = str(
        final_registered_model.version
    )
    final_results_manifest["mlflow"]["registered_model_uri"] = (
        f"models:/{FINAL_REGISTERED_MODEL_NAME}/{final_registered_model.version}"
    )

    with open(LOCAL_FINAL_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(final_results_manifest, f, indent=2)

    upload_local_to_s3(
        LOCAL_FINAL_RESULTS_PATH,
        FINAL_RESULTS_S3_URI,
    )
    mlflow.log_artifact(
        str(LOCAL_FINAL_RESULTS_PATH),
        artifact_path="modela_outputs",
    )



✓ Saved to S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_xgboost_final.json
✓ Final model saved locally and to S3: model_a_xgboost_final.json


Registered model 'ITI113-team04-ModelA-XGBoost-Final' already exists. Creating a new version of this model...
2026/08/18 06:12:26 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ITI113-team04-ModelA-XGBoost-Final, version 5
Created version '5' of model 'ITI113-team04-ModelA-XGBoost-Final'.


✓ Final model logged and registered.
Run ID: 637cb783c2be4269a50132c206cd36a5
Registered model: ITI113-team04-ModelA-XGBoost-Final
Version: 5
Alias: champion-candidate → 5
✓ Saved to S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/model_a_final_results.json
🏃 View run S402_MODELA_FINAL at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/4/runs/637cb783c2be4269a50132c206cd36a5
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/4


## 20. Registered-Model Smoke Test

The final model is loaded through the exact MLflow Model Registry URI
`models:/<registered-model>/<version>`, rather than only through the run
artifact URI.

The smoke test compares predictions from the registry-loaded model against the
in-memory final model on identical model-ready inputs. Passing this check
confirms that the exact registered version reproduces the final model
probabilities.

This test does not perform threshold selection, test-set optimisation or any
additional model fitting.


In [27]:
# ============================================================
# Registry smoke test — validate the registered-model URI
# ============================================================
REGISTRY_MODEL_URI = (
    f"models:/{FINAL_REGISTERED_MODEL_NAME}/{final_registered_model.version}"
)

loaded_registered_model = mlflow.xgboost.load_model(
    REGISTRY_MODEL_URI
)

smoke_input = X_test_final_model.head(20)
direct_prob = final_xgb_model.predict_proba(smoke_input)[:, 1]
registered_prob = loaded_registered_model.predict_proba(smoke_input)[:, 1]

if not np.allclose(
    direct_prob,
    registered_prob,
    rtol=1e-6,
    atol=1e-8,
):
    raise ValueError(
        "Registry-loaded Model A probabilities do not match the "
        "in-memory final model."
    )

print("✓ Registered-model URI smoke test passed.")
print(f"✓ Registry URI: {REGISTRY_MODEL_URI}")
print("✓ Registry-loaded model reproduces the final model probabilities.")


✓ Registered-model URI smoke test passed.
✓ Registry URI: models:/ITI113-team04-ModelA-XGBoost-Final/5
✓ Registry-loaded model reproduces the final model probabilities.


## 21. Final Model A Summary

Model A is now finalised as a reproducible XGBoost + SMOTE candidate:

1. the milestone baseline is retained as the benchmark;
2. candidate configurations and operating thresholds are selected using
   validation data only;
3. the selected configuration is retrained on the combined development data;
4. the untouched test set is evaluated once;
5. final fairness, feature-importance, comparison and deployment artefacts
   are exported;
6. the final model is logged and registered in MLflow; and
7. the exact registered model version is verified through a registry smoke test.

### Final delivery artefacts

- milestone baseline model and results;
- final XGBoost model;
- final candidate comparison CSV;
- baseline-versus-final comparison;
- final XGBoost Gain importance;
- final CART Gini cross-check;
- final fairness test-prediction parquet;
- deployment contract;
- preprocessing bundle;
- final machine-readable results manifest;
- MLflow signature and input example;
- registered model version and `champion-candidate` alias;
- registered-model URI smoke-test evidence.

### Governance status

Model A is a **final Model A candidate**, not yet the project champion.
Model B must still be evaluated and compared using the group's agreed common
evaluation basis.

The Fairness notebook should consume the final prediction export
`model_a_final_fairness_test_predictions.parquet` when auditing the final
Model A candidate, rather than reconstructing or altering the test split.


In [30]:
bucket = "nyp-26s1-iti113"
prefix = "iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/"

boto3.client("s3").download_file(bucket, prefix + "model_a_xgboost_final.json", "model_dir/xgboost_model.json")
boto3.client("s3").download_file(bucket, prefix + "model_a_deployment_contract.json", "model_dir/model_a_deployment_contract.json")
boto3.client("s3").download_file(bucket, prefix + "model_a_preprocessing_bundle.joblib", "model_dir/model_a_preprocessing_bundle.joblib")